# Generate one year of demand based on calendar

In [1]:
import holidays
import yaml

from copy import deepcopy
from datetime import date, timedelta

In [2]:
with open('../data/MAD-BCN/demand_data.yaml', 'r') as f:
    demand_yaml = f.read()

data = yaml.safe_load(demand_yaml)

In [ ]:
class Calendar1(holidays.HolidayBase): 
    # Monday-Thursday-LowDemand

    def _populate(self, year):
        # Febraury
        for i in range(1, 29):
            self[date(year, 2, i)] = 'Holidays'
        # March - October
        for i in range(1, 32):
            self[date(year, 3, i)] = 'Holidays'
            self[date(year, 10, i)] = 'Holidays'
        # June - September - November
        for i in range(1, 31):
            self[date(year, 6, i)] = 'Holidays'
            self[date(year, 9, i)] = 'Holidays'
            self[date(year, 11, i)] = 'Holidays'

In [ ]:
class Calendar2(holidays.HolidayBase): 
    # Monday-Thursday-HighDemand

    def _populate(self, year):
        # January - May - July - August - December
        for i in range(1, 32):
            self[date(year, 1, i)] = 'Holidays'
            self[date(year, 5, i)] = 'Holidays'
            self[date(year, 7, i)] = 'Holidays'
            self[date(year, 8, i)] = 'Holidays'
            self[date(year, 12, i)] = 'Holidays'
        # April
        for i in range(1, 31):
            self[date(year, 4, i)] = 'Holidays'

In [ ]:
class CalendarMadrid(holidays.HolidayBase): 
    # Madrid Holidays
    
    def _populate(self, year):
        self[date(year, 5, 2)] = 'Holidays'
        self[date(year, 5, 15)] = 'Holidays'
        self[date(year, 11, 10)] = 'Holidays'

In [ ]:
class CalendarBCN(holidays.HolidayBase): 
    # Barcelona Holidays
    
    def _populate(self, year):
        self[date(year, 4, 21)] = 'Holidays'
        self[date(year, 6, 24)] = 'Holidays'
        self[date(year, 9, 11)] = 'Holidays'
        self[date(year, 12, 26)] = 'Holidays'

In [ ]:
class CalendarOrange(holidays.HolidayBase):
    # Days with Orange Warning in Spain (really hight temperatures)
    # https://www.aemet.es/es/eltiempo/prediccion/avisos?w=0&l=0&datos=avisos

    def _populate(self, year):
        self[date(year, 5, 24)] = 'Orange Warning'
        self[date(year, 5, 25)] = 'Orange Warning'
        self[date(year, 9, 13)] = 'Orange Warning'
        self[date(year, 9, 14)] = 'Orange Warning'

In [ ]:
from holidays.countries import ES 

class CustomHolidays(ES):
    def _populate(self, year):
        super()._populate(year)
        for i in range(1, 7):
            self[date(year, 1, i)] = 'Navidad'
        self[date(year, 2, 14)] = 'San Valentín'
        for i in range(27, 29):
            self[date(year, 2, i)] = 'Carnaval'
        for i in range(1, 6):
            self[date(year, 3, i)] = 'Carnaval'
        for i in range(12, 22):
            self[date(year, 4, i)] = 'Semana Santa'

        self[date(year, 5, 1)] = 'Fista del Trabajo'

        for i in range(31, 32):
            self[date(year, 7, i)] = 'Verano'
        for i in range(1, 4):
            self[date(year, 8, i)] = 'Verano'
        for i in range(14, 18):
            self[date(year, 8, i)] = 'Verano'
        for i in range(29, 32):
            self[date(year, 8, i)] = 'Verano'

        self[date(year, 11, 1)] = 'Todos los santos'
        for i in range(24, 26):
            self[date(year, 12, i)] = 'Navidad'

In [ ]:
es_holidays = holidays.ES()
madrid_holidays = CalendarMadrid()
BCN_holidays = CalendarBCN()
calendar1 = Calendar1()
calendar2 = Calendar2()
calendar_orange = CalendarOrange()

def get_demand_pattern(_date: date):
    demand_pattern = None
    if _date.weekday() >= 0 and _date.weekday() <= 3:
        if _date in calendar1:
            demand_pattern = 1 # Monday-Thrusday-LowDemand
        elif _date in calendar2:
            demand_pattern = 3 # Monday-Thrusday-HighDemand
    elif _date.weekday() > 3 and _date.weekday() <= 6:
        if _date in calendar1:
            demand_pattern = 2 # Weekend-LowDemand
        elif _date in calendar2:
            demand_pattern = 4 # Weekend-HighDemand
    if _date in es_holidays:
        demand_pattern = 4 # Weekend-2
    if _date in madrid_holidays or _date in BCN_holidays:
        demand_pattern = 5 # Local-Regional-Holidays
    if _date in calendar_orange:
        demand_pattern = 6 # Orange-Warning
    return demand_pattern

In [10]:
start_date = date(2025, 1, 1)
end_date = date(2025, 12, 31)

# Initialize the first day
data['day'] = []
data['day'].append({})
# Set the first day
data['day'][0]['id'] = 1
data['day'][0]['date'] = start_date
data['day'][0]['demandPattern'] = get_demand_pattern(start_date)

# Add the rest of the days
while start_date < end_date:
    previous_day = deepcopy(data['day'][-1])
    previous_day['id'] += 1
    previous_day['date'] += timedelta(days=1)
    previous_day['demandPattern'] = get_demand_pattern(previous_day['date'])
    data['day'].append(previous_day)
    start_date += timedelta(days=1)

In [11]:
data

{'market': [{'id': 1,
   'departure_station': '60000',
   'arrival_station': '71801'}],
 'userPattern': [{'id': 1,
   'name': 'Business',
   'arrival_time': 'norm',
   'arrival_time_kwargs': {'loc': 8, 'scale': 1},
   'purchase_day': 'poisson',
   'purchase_day_kwargs': {'mu': 1.25},
   'forbidden_departure_hours': {'start': 9, 'end': 24},
   'seats': [{'id': 1, 'utility': 10},
    {'id': 2, 'utility': 15},
    {'id': 3, 'utility': 18}],
   'train_service_providers': [{'id': 1, 'utility': 0}],
   'penalty_arrival_time': 'polynomial',
   'penalty_arrival_time_kwargs': {'b_0': 0, 'b_1': 0.8},
   'penalty_departure_time': 'polynomial',
   'penalty_departure_time_kwargs': {'b_0': 0, 'b_1': 0.7},
   'penalty_cost': 'polynomial',
   'penalty_cost_kwargs': {'b_0': 0, 'b_1': 0.05},
   'penalty_travel_time': 'polynomial',
   'penalty_travel_time_kwargs': {'b_0': 0, 'b_1': 0.4},
   'error': 'norm',
   'error_kwargs': {'loc': 2, 'scale': 1}},
  {'id': 2,
   'name': 'Student',
   'arrival_time': '

In [12]:
yaml.safe_dump(data, open('../data/MAD-BCN/demand_data.yaml', 'w'), sort_keys=False, allow_unicode=True)